In [77]:
%useLatestDescriptors
%use dataframe
@file:DependsOn("com.github.doyaaaaaken:kotlin-csv-jvm:1.7.0")

import com.github.doyaaaaaken.kotlincsv.dsl.csvWriter
import io.github.oshai.kotlinlogging.KotlinLogging.logger
import kotlin.reflect.full.declaredMemberProperties
import java.nio.file.Paths
import java.util.Locale
import kotlin.io.path.Path

enum class Mode { FLAT, RANDOM }
enum class Algorithm(val shortName: String) {
    FROMBACK("tsprcs"),
    DISTANCE("tsprce"),
    SPARSITY("tsprcs"),
    OP("op"),
}

enum class Context { ELIMINATION, BUDGET, CLUSTERING }

val percentageFraction = 1
val colsWithoutPercentages = "cdml"
val gradient = 0.1

val mode = Mode.FLAT
val algorithm = Algorithm.OP
val context = Context.BUDGET

val fileName = "budget_comparison_cd.csv"//"comparison_${mode.name.lowercase()}.csv"

val relativePath = "/op-solver-strict/results/${context.name.lowercase()}/comparison/"
val navigationPath = Paths.get(System.getProperty("user.dir"), "../../../..").normalize().toAbsolutePath().toString()

val path = Paths.get(navigationPath, relativePath, fileName).toString()

var df = DataFrame.readCsv(path)
df

instance,cdmlmx_0.2,cdml,cdcmx_0.2,cdc
eil101,3251,2884,3269,3184
gil262,7408,6841,7191,7139
pr299,8598,7202,8332,7509
lin318,9846,8178,10031,9272
rd400,11139,10515,11025,11187
d493,16012,13936,15942,14734
u574,16793,15277,16792,15902
u724,20681,18573,21171,20050
pcb1173,32005,29045,31964,27810
fl1400,48561,43182,50667,46541


In [78]:
val bestValues = df.convert { all() }.perRowCol { row, col ->
    if (col[row] is String) {
        0
    } else {
        (col[row] as Number).toInt()
    }
}.map { row ->
    row.rowMaxOf<Int>()
}

bestValues

[3269, 7408, 8598, 10031, 11187, 16012, 16793, 21171, 32005, 50667, 65701]

In [79]:

 //max(maxRevenueDif,0.0)

val rowMaxValues = bestValues.mapIndexed { index, resultMax -> max(resultMax, (df[colsWithoutPercentages][index] as Number).toInt()) }

rowMaxValues

[3269, 7408, 8598, 10031, 11187, 16012, 16793, 21171, 32005, 50667, 65701]

In [80]:
val rowMinValues = df.map { row ->
    row.rowMinOfOrNull<Int>()
}.map { row -> row!!.toInt()}
rowMinValues

[2884, 6841, 7202, 8178, 10515, 13936, 15277, 18573, 27810, 43182, 57944]

In [81]:

val revenueDif = rowMaxValues.mapIndexed { index, maxEntry ->
    val minEntry = df[index].rowMinOf<Int>()
    (maxEntry - minEntry!!).toDouble() / maxEntry.toDouble()
}

fun getSaturation(gradient: Double, maxValue: Int, value: Int): String {
    return min(max((100 - ((maxValue - value) / (maxValue * gradient) * 100)), 0.0), 100.0).toInt().toString()
}

fun calculatePercentage(refValue: Int, compValue: Int): Double {
    return ((compValue.toDouble() - refValue.toDouble()) / refValue.toDouble())
}

fun formatePercentage(value: Double): String {
    return "${String.format(Locale.US, "%+.${percentageFraction}f", value * 100)}\\%"
}

fun getPercentage(row: DataRow<*>, compValue: Int): String {
    val refValue = (df.get(colsWithoutPercentages)[row] as Number).toInt()
    val percentage = calculatePercentage(refValue, compValue)
    return "{\\tiny${formatePercentage(percentage)}}"
}

val footer = df.convert { all() }.perRowCol { row, col ->
    if (col.name() == colsWithoutPercentages || col[row] is String) {
        10000.0
    } else {
        val refValue = (df.get(colsWithoutPercentages)[row] as Number).toInt()
        calculatePercentage(refValue, (col[row] as Number).toInt())
    }
}.mean().values().mapIndexed { index, it ->
    if (index == 0) {
        "avg diff"
    } else if (it is Number && it.toDouble() > 100.0) {
        "-"
    } else if (it is Number) {
        formatePercentage(it.toDouble())
    } else {
        "${it.toString()}\\%"
    }
}.toList()
footer

[avg diff, +12.6\%, -, +12.7\%, +5.9\%]

In [82]:

val stringdf = df.convert { all() }.perRowCol { row, col ->
    if (col[row] is String) {
        col[row].toString().split("-").first()
    } else if (col.name() == colsWithoutPercentages) {
        val value = (col[row] as Number).toInt()
        val maxValue = rowMaxValues[row.index()]
        val saturation = getSaturation(gradient, maxValue, value)
        if (bestValues[row.index()] == value) {
            "\\cellcolor{cyan!$saturation} \\textbf{$value*}"
        } else {
            "\\cellcolor{cyan!$saturation} $value"
        }
    } else {
        val value = (col[row] as Number).toInt()
        val maxValue = rowMaxValues[row.index()]
        val saturation = getSaturation(gradient, maxValue, value)
        val percentage = getPercentage(row, value)
        if (bestValues[row.index()] == value) {
            "\\cellcolor{cyan!$saturation} \\textbf{$value*}$percentage"
        } else {
            "\\cellcolor{cyan!$saturation} $value$percentage"
        }
    }
}
stringdf

instance,cdmlmx_0.2,cdml,cdcmx_0.2,cdc
eil101,\cellcolor{cyan!94} 3251{\tiny+12.7\%},\cellcolor{cyan!0} 2884,\cellcolor{cyan!100} \textbf{3269*}{\...,\cellcolor{cyan!73} 3184{\tiny+10.4\%}
gil262,\cellcolor{cyan!100} \textbf{7408*}{\...,\cellcolor{cyan!23} 6841,\cellcolor{cyan!70} 7191{\tiny+5.1\%},\cellcolor{cyan!63} 7139{\tiny+4.4\%}
pr299,\cellcolor{cyan!100} \textbf{8598*}{\...,\cellcolor{cyan!0} 7202,\cellcolor{cyan!69} 8332{\tiny+15.7\%},\cellcolor{cyan!0} 7509{\tiny+4.3\%}
lin318,\cellcolor{cyan!81} 9846{\tiny+20.4\%},\cellcolor{cyan!0} 8178,\cellcolor{cyan!100} \textbf{10031*}{...,\cellcolor{cyan!24} 9272{\tiny+13.4\%}
rd400,\cellcolor{cyan!95} 11139{\tiny+5.9\%},\cellcolor{cyan!39} 10515,\cellcolor{cyan!85} 11025{\tiny+4.9\%},\cellcolor{cyan!100} \textbf{11187*}{...
d493,\cellcolor{cyan!100} \textbf{16012*}{...,\cellcolor{cyan!0} 13936,\cellcolor{cyan!95} 15942{\tiny+14.4\%},\cellcolor{cyan!20} 14734{\tiny+5.7\%}
u574,\cellcolor{cyan!100} \textbf{16793*}{...,\cellcolor{cyan!9} 15277,\cellcolor{cyan!99} 16792{\tiny+9.9\%},\cellcolor{cyan!46} 15902{\tiny+4.1\%}
u724,\cellcolor{cyan!76} 20681{\tiny+11.3\%},\cellcolor{cyan!0} 18573,\cellcolor{cyan!100} \textbf{21171*}{...,\cellcolor{cyan!47} 20050{\tiny+8.0\%}
pcb1173,\cellcolor{cyan!100} \textbf{32005*}{...,\cellcolor{cyan!7} 29045,\cellcolor{cyan!98} 31964{\tiny+10.0\%},\cellcolor{cyan!0} 27810{\tiny-4.3\%}
fl1400,\cellcolor{cyan!58} 48561{\tiny+12.5\%},\cellcolor{cyan!0} 43182,\cellcolor{cyan!100} \textbf{50667*}{...,\cellcolor{cyan!18} 46541{\tiny+7.8\%}


In [83]:
fun getLatexTable(formating: String, amountColumns: String, title: String, header: String, label: String, caption: String, body: String): String {
    return """
    \begin{table}[]
        \vspace{2em}
        \begin{adjustbox}{center}
            \begin{tabular}{ $formating  }
                \hline
                \multicolumn{$amountColumns}{|c|}{$title} \\
                \hline
                    $header \\
                \hline
                    $body
                \hline
            \end{tabular}
        \end{adjustbox}
        \caption{$caption}
        \label{$label}
    \end{table}
         """
}

val shortAlgString = when (algorithm) {
    Algorithm.FROMBACK -> "TSPrfb"
    Algorithm.DISTANCE -> "TSPrce"
    Algorithm.SPARSITY -> "TSPrcs"
    Algorithm.OP -> "OP"
}
val mediumAlgString = when (algorithm) {
    Algorithm.FROMBACK -> "cluster removal from back"
    Algorithm.DISTANCE -> "cluster removal based on distance"
    Algorithm.SPARSITY -> "cluster removal based on sparsity"
    Algorithm.OP -> "implicit cluster removal"
}

val amountColumns = df.columns().size.toString()
val formating = "|"+ df.columns().joinToString(separator = "") { "p{1.9cm}|" }
val header = df.columnNames().joinToString(separator = " & ")
val label = "tab:elim:${mode.name.lowercase()}"
val title = "emimination methods ${mode.name.lowercase()}."
//Parameter run for $R'$ using cluster removal from back with a budget of $\gamma = 0.5$. The percentage value refers to the mean revenue increase compared to $e^{blr}$. The highest revenue of an instance has 100\% saturation decreasing to 0\% at 70\% of the maximum. $\textbf{*}$ refers to the best mean revenue.
val caption = "Emimination method comparison for \$R' = 0.5\$ and \$\\alpha = 0.25\$ with $\\gamma = 0.5\$. The percentage value refers to the mean revenue increase compared to \$e^{bl${mode.name.lowercase().first().toString()}}$. The highest revenue of an instance has 100\\% saturation decreasing to 0\\% at ${(100 + gradient * -100).toInt()}\\% of the maximum revenue. The larges revenue value is referenced by \$\\textbf{*}\$."

val body = stringdf.rows().joinToString(separator = " \\\\ \n") { row ->
    row.values().joinToString(separator = " & ") {
        it.toString()
    }
} + " \\\\ \\hline " + footer.joinToString(separator = " & ") {
    it.toString()
} + " \\\\"

getLatexTable(formating, amountColumns, title, header, label, caption, body)


    \begin{table}[]
        \vspace{2em}
        \begin{adjustbox}{center}
            \begin{tabular}{ |p{1.9cm}|p{1.9cm}|p{1.9cm}|p{1.9cm}|p{1.9cm}|  }
                \hline
                \multicolumn{5}{|c|}{emimination methods flat.} \\
                \hline
                    instance & cdmlmx_0.2 & cdml & cdcmx_0.2 & cdc \\
                \hline
                    eil101 & \cellcolor{cyan!94} 3251{\tiny+12.7\%} & \cellcolor{cyan!0} 2884 & \cellcolor{cyan!100} \textbf{3269*}{\tiny+13.3\%} & \cellcolor{cyan!73} 3184{\tiny+10.4\%} \\ 
gil262 & \cellcolor{cyan!100} \textbf{7408*}{\tiny+8.3\%} & \cellcolor{cyan!23} 6841 & \cellcolor{cyan!70} 7191{\tiny+5.1\%} & \cellcolor{cyan!63} 7139{\tiny+4.4\%} \\ 
pr299 & \cellcolor{cyan!100} \textbf{8598*}{\tiny+19.4\%} & \cellcolor{cyan!0} 7202 & \cellcolor{cyan!69} 8332{\tiny+15.7\%} & \cellcolor{cyan!0} 7509{\tiny+4.3\%} \\ 
lin318 & \cellcolor{cyan!81} 9846{\tiny+20.4\%} & \cellcolor{cyan!0} 8178 & \cellcolor{cyan!100} \textbf{10031*